In [0]:
# create schemas (run once)

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Setup completed")

In [0]:
print("Bronze started")

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .load("/Volumes/workspace/default/data/Sample_ Superstore.csv")

df_clean = df.toDF(*[c.replace(" ", "_").replace("-", "_") for c in df.columns])

# write Bronze
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.superstore")

print("Bronze completed")

In [0]:
print("Silver started (incremental)")

from pyspark.sql.functions import col, to_date, coalesce, regexp_replace

# Bronze (new data)
bronze_df = spark.table("workspace.bronze.superstore")

display(bronze_df)

# Silver (existing data)
try:
    silver_df = spark.table("workspace.silver.superstore")
except:
    silver_df = None


# ✅ Step 1: Find new records
if silver_df is not None and 'Order_ID' in silver_df.columns and 'Product_Name' in silver_df.columns:
    new_data = bronze_df.join(silver_df, on=['Order_ID', 'Product_Name'], how='left_anti')
else:
    new_data = bronze_df
    
new_data.printSchema()
# ✅ Step 2: Clean only new data
new_data = new_data \
    .withColumn(
    "Order_Date",
    to_date(
        regexp_replace(col("Order_Date"), "-", "/"),
        "M/d/yyyy"
    )
).withColumn(
    "Ship_Date",
    to_date(
        regexp_replace(col("Ship_Date"), "-", "/"),
        "M/d/yyyy"
    )
) \
    .dropna(subset=["Order_ID"]) \
    .fillna({
        "Sales": 0,
        "Profit": 0,
        "Discount": 0
    }) \
    .dropDuplicates()

display(new_data)    

#spark.sql("drop table if exists workspace.silver.superstore")
# ✅ Step 3: Append (not overwrite)
new_data.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.silver.superstore")

print("Silver incremental completed")

In [0]:
silver_df = spark.table("workspace.silver.superstore")
display(silver_df)
print("Gold started")

from pyspark.sql.functions import col, sum, avg, countDistinct

# Silver
silver_df = spark.table("workspace.silver.superstore")
# Gold
gold_df = silver_df.groupBy("Ship_Mode").agg(
    sum(col("Sales")).alias("Total_Sales"),
    avg(col("Profit")).alias("Total_Profit"),
    countDistinct(col("Order_ID")).alias("Total_Orders")
)

display(gold_df)


gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.superstore_sales_by_ship_mode")
print("Gold finished")